# Playlist Sheet

One playlist, one row per track, shaped to paste straight into Excel and annotate by hand.

Set the playlist with `make report-06 PLAYLIST="<name>"`.


## 1. Connect

The profile scopes the play counts in §3 — they are *this* person's listening, not the household's.


In [ ]:
import os

from IPython.display import Markdown

from spotify_lakehouse import playlist_sheet
from spotify_lakehouse.notebook import setup

ctx = setup()
PLAYLIST = os.environ.get(playlist_sheet.PLAYLIST_ENV) or playlist_sheet.DEFAULT_PLAYLIST
display(
    Markdown(
        f"Playlist **`{PLAYLIST}`** · profile **`{ctx.profile}`** · "
        f"schemas `{ctx.stg_schema}` and `{ctx.mart_schema}`."
    )
)

## 2. Does the playlist resolve?

A name that matches nothing, and a name that matches a playlist with **nothing loaded**, are
different problems and read differently below. A playlist Spotify refuses (403) or one created since
the last `make playlists` run has a row but no tracks.

<!-- caption: Current playlists matching the requested name, with how many of their tracks are loaded -->


In [ ]:
sheet = playlist_sheet.sheet(ctx, PLAYLIST)
ready = not sheet.empty
if ready:
    display(Markdown(f"**{len(sheet)}** tracks loaded for **`{PLAYLIST}`**."))
else:
    display(
        Markdown(
            f"> [!WARNING]\n"
            f"> **`{PLAYLIST}` has no loaded tracks.** Either the name does not match, or it "
            f"matches a playlist whose contents were never ingested. The closest current "
            f"playlists are below — `loaded` is the column that matters.\n"
            f">\n"
            f"> If the playlist is newer than the last ingest, run "
            f"`make playlists PROFILE={ctx.profile}` (it calls the Spotify API)."
        )
    )
    display(playlist_sheet.candidates(ctx, PLAYLIST))

## 3. The sheet

Playlist order, one row per track. **`key`, `bpm` and `notes` are deliberately empty** — Spotify's
`/audio-features` is cut off for this app, so nothing here can fill them, and annotating them is the
point of the sheet.

`duration` is text so Excel will not reinterpret it as a date; `duration_ms` and `minutes` are
numbers to sort and sum on.

<!-- caption: Every track in the playlist, in playlist order, with what the warehouse knows about it -->


In [ ]:
if ready:
    display(sheet)
else:
    display(Markdown("> [!WARNING]\n> Not run: see §2."))

## 4. What is actually populated

🚨 **Read this before building on a column.** A column that exists is not a column that has data.
The API-resolved metadata — `album`, `year`, `explicit`, `track_no`, `album_type` — only exists for
tracks that have had a `GET /tracks/{id}` lookup, and that is usually a minority. The play columns
are the densest thing here.

`spot resolve-tracks --run` fills the sparse ones in at roughly one API call per second.

<!-- caption: Populated count per column, computed from the sheet above rather than assumed -->


In [ ]:
if ready:
    display(playlist_sheet.coverage(sheet))
else:
    display(Markdown("> [!WARNING]\n> Not run: see §2."))

## 5. The file to open in Excel

Written to `published/`, which is gitignored — nothing about this playlist reaches the public repo.
Open it directly in Excel rather than copying out of the table above, which preserves the column
types.


In [ ]:
if ready:
    from spotify_lakehouse.config import repo_root

    written = playlist_sheet.write_sheet(sheet, PLAYLIST, repo_root() / "published")
    display(
        Markdown(
            f"**{len(sheet)}** rows written to `{written.relative_to(repo_root())}`.\n\n"
            f"Open with: `open {written.relative_to(repo_root())}`"
        )
    )
else:
    display(Markdown("> [!WARNING]\n> Nothing written: see §2."))

The warehouse connection is closed at the end of the run.

In [ ]:
ctx.close()